# Pipeline check: CheXzero vs CLEAR on the Indiana University chest X-rays

**Why this exists.** On NIH ChestX-ray14, CLEAR's concept-based zero-shot scored below CheXzero (macro-AUROC 0.635 vs 0.695), although CLEAR's paper reports CLEAR ahead of CheXzero on Indiana, VinDr-CXR and PadChest. Indiana is openly available, so this notebook runs exactly the same pipeline on it and asks one question: **does CLEAR beat CheXzero on Indiana with our pipeline, as the paper reports?**

**Decision rule, fixed before running (1 October 2026 plan, written 2 October 2026):**
- Primary estimate: Delta = macro-AUROC(CLEAR concept, plain prompts, padded images) minus macro-AUROC(CheXzero ensemble), over all eligible findings and all images, with a 95% interval from 1,000 report-level bootstrap resamples (seed 2026).
- **Consistent with the CLEAR paper** if Delta > 0 and CLEAR wins more findings than it loses.
- **Inconsistent with the CLEAR paper** if Delta < 0 and the 95% interval excludes 0.
- **Inconclusive** otherwise.
- Findings: every term in the Indiana "Problems" field except "normal", "No Indexing" and "Technical Quality of Image Unsatisfactory", kept if at least 20 reports are positive. Prompts are the lower-cased terms, the same for both models.
- Secondary: frontal images only; CLEAR direct scoring; CLEAR with instructed prompts; CLEAR with the package's squash-resize instead of padding.

**Settings:** Accelerator **GPU T4 x2**, Internet **On**, inputs **Chest X-rays (Indiana University)** (raddar) and your private **chexzero-weights** dataset.
**Run:** Save Version, then Save & Run All (Commit). At the end, copy the block between `===== INDIANA SUMMARY =====` and `===== END =====`.

In [ ]:
# 0. Settings
import os, sys, time, json, glob, subprocess, hashlib, platform
os.environ["HF_HOME"] = "/tmp/hf"          # large downloads go to /tmp, not the 20 GB output folder
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
SEED = 2026
OUT = "/kaggle/working/indiana_out"; os.makedirs(OUT, exist_ok=True)
FINDINGS_NIH = ["Atelectasis","Cardiomegaly","Effusion","Infiltration","Mass","Nodule","Pneumonia",
            "Pneumothorax","Consolidation","Edema","Emphysema","Fibrosis","Pleural_Thickening","Hernia"]
PROMPT_WORDS = {"Atelectasis":"atelectasis","Cardiomegaly":"cardiomegaly","Effusion":"pleural effusion",
                "Infiltration":"infiltration","Mass":"mass","Nodule":"nodule","Pneumonia":"pneumonia",
                "Pneumothorax":"pneumothorax","Consolidation":"consolidation","Edema":"edema",
                "Emphysema":"emphysema","Fibrosis":"fibrosis","Pleural_Thickening":"pleural thickening",
                "Hernia":"hernia"}
run_info = {"started_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()), "seed": SEED,
            }
T0 = time.time()

In [ ]:
# 1. Packages and versions
!pip install -q git+https://github.com/openai/CLIP.git gdown
!rm -rf /kaggle/working/CLEAR && git clone -q https://github.com/peterhan91/CLEAR.git /kaggle/working/CLEAR
!pip install -q /kaggle/working/CLEAR
import torch, numpy as np, pandas as pd, transformers
run_info["clear_git_commit"] = subprocess.run(["git","-C","/kaggle/working/CLEAR","rev-parse","HEAD"],
                                              capture_output=True, text=True).stdout.strip()
try:
    from huggingface_hub import HfApi
    run_info["clear_hf_sha"] = HfApi().model_info("peterhan91/CLEAR").sha
except Exception as e:
    run_info["clear_hf_sha"] = "unavailable: " + repr(e)[:200]
run_info.update({"python": platform.python_version(), "torch": torch.__version__,
                 "transformers": transformers.__version__,
                 "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]})
print(json.dumps(run_info, indent=1))
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2"

In [ ]:
# 2. Indiana cohort and labels from the "Problems" field
def find_file(name):
    for root, dirs, files in os.walk("/kaggle/input"):
        if name in files:
            return os.path.join(root, name)
    raise FileNotFoundError(name + " not found: is the Indiana dataset attached?")
rep = pd.read_csv(find_file("indiana_reports.csv"))
proj = pd.read_csv(find_file("indiana_projections.csv"))
print("report columns:", list(rep.columns)); print("projection columns:", list(proj.columns))
assert {"uid", "Problems"} <= set(rep.columns) and {"uid", "filename", "projection"} <= set(proj.columns)
png = {}
for root, dirs, files in os.walk(os.path.dirname(find_file("indiana_reports.csv"))):
    for f in files:
        if f.lower().endswith(".png"): png[f] = os.path.join(root, f)
proj["path"] = proj.filename.map(png)
print("images listed:", len(proj), "| found on disk:", int(proj.path.notna().sum()))
df = proj.dropna(subset=["path"]).merge(rep[["uid", "Problems"]], on="uid", how="inner")
EXCLUDE = {"normal", "no indexing", "technical quality of image unsatisfactory"}
def terms(s):
    if not isinstance(s, str): return set()
    return {t.strip() for t in s.split(";") if t.strip() and t.strip().lower() not in EXCLUDE}
df["terms"] = df.Problems.map(terms)
report_terms = rep.set_index("uid").Problems.map(terms)
counts = pd.Series([t for ts in report_terms for t in ts]).value_counts()
FINDINGS = [t for t, n in counts.items() if n >= 20]
for f in FINDINGS:
    df[f] = df.terms.map(lambda ts: int(f in ts))
POS = [f.lower() for f in FINDINGS]
NEG = ["no " + w for w in POS]
run_info.update({"prompts_pos": POS, "prompts_neg": NEG, "n_images": int(len(df)), "n_reports": int(df.uid.nunique()),
                 "n_frontal": int((df.projection == "Frontal").sum()),
                 "findings_positive_reports": {f: int(counts[f]) for f in FINDINGS}})
print(len(FINDINGS), "findings with at least 20 positive reports:")
print(counts[counts >= 20].to_string())
cohort = df.reset_index(drop=True)
cohort.drop(columns=["path", "terms"]).to_csv(f"{OUT}/cohort.csv", index=False)

In [ ]:
# 4. Prompt embeddings with SFR-Embedding-Mistral, following CLEAR's get_embed.py
#    (AutoModel, fp16, last-token pooling, L2 normalisation). This runs in a separate process so
#    that all its GPU memory is released when it finishes. Each text is embedded on its own (no padding).
#    Two prompt versions are saved: plain text, as in the authors' exp_zeroshot.py (primary), and with
#    the instruction prefix that reproduces the authors' released concept embeddings (secondary).
import gc
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from clear.concepts import ConceptBank
concepts_csv = hf_hub_download("peterhan91/CLEAR", "mimic_concepts.csv")
clip_pt = hf_hub_download("peterhan91/CLEAR", "concept_embeddings_368294.pt")
llm_pkl = hf_hub_download("peterhan91/CLEAR", "concepts_embeddings_sfr_mistral.pickle")
bank = ConceptBank.from_files(concepts_csv, clip_embeddings_path=clip_pt, llm_embeddings_path=llm_pkl,
                              allow_unsafe_pickle=True)
C_llm = bank.ensure_llm_embeddings()
pick = np.random.default_rng(SEED).choice(bank.num_concepts, 40, replace=False)
INSTR = "Instruct: Given medical radiology concepts, retrieve relevant passages and generate embeddings\nQuery: "
json.dump({"pos": POS, "neg": NEG, "check": [bank.concepts[i] for i in pick], "instr": INSTR},
          open("/kaggle/working/sfr_in.json", "w"))
SFR_SCRIPT = r'''
import json, torch, numpy as np, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
d = json.load(open("/kaggle/working/sfr_in.json"))
tok = AutoTokenizer.from_pretrained("Salesforce/SFR-Embedding-Mistral")
mm = {i: "13GiB" for i in range(torch.cuda.device_count())}; mm["cpu"] = "24GiB"
llm = AutoModel.from_pretrained("Salesforce/SFR-Embedding-Mistral", dtype=torch.float16,
                                device_map="auto", max_memory=mm).eval()
dev0 = next(llm.parameters()).device
def pool(h, m):
    if m[:, -1].sum() == m.shape[0]:
        return h[:, -1]
    return h[torch.arange(h.shape[0], device=h.device), m.sum(1) - 1]
@torch.no_grad()
def emb(texts):
    out = []
    for s in texts:
        e = tok(s, max_length=8192, truncation=True, return_tensors="pt").to(dev0)
        out.append(F.normalize(pool(llm(**e).last_hidden_state, e["attention_mask"]).float(), dim=-1).cpu())
    return torch.cat(out).numpy()
I = d["instr"]
np.savez("/kaggle/working/sfr_out.npz",
         pos=emb(d["pos"]), neg=emb(d["neg"]),
         pos_i=emb([I + s for s in d["pos"]]), neg_i=emb([I + s for s in d["neg"]]),
         chk=emb(d["check"]), chk_i=emb([I + s for s in d["check"]]))
print("SFR embeddings done")
'''
open("/kaggle/working/sfr_embed.py", "w").write(SFR_SCRIPT)
t = time.time()
r = subprocess.run([sys.executable, "/kaggle/working/sfr_embed.py"], capture_output=True, text=True)
print(r.stdout[-1500:]); print(r.stderr[-3000:])
assert r.returncode == 0, "SFR embedding step failed"
z = np.load("/kaggle/working/sfr_out.npz")
pos_llm, neg_llm, pos_llm_i, neg_llm_i = z["pos"], z["neg"], z["pos_i"], z["neg_i"]
for k in ["pos", "neg", "pos_i", "neg_i"]:
    np.save(f"{OUT}/prompt_emb_sfr_{k}.npy", z[k])
ref = C_llm[pick] / np.linalg.norm(C_llm[pick], axis=1, keepdims=True)
cos_plain = (z["chk"] * ref).sum(1); cos_instr = (z["chk_i"] * ref).sum(1)
run_info["sfr_check_cos_plain_mean_min"] = [float(cos_plain.mean()), float(cos_plain.min())]
run_info["sfr_check_cos_instruct_mean_min"] = [float(cos_instr.mean()), float(cos_instr.min())]
run_info["n_concepts"] = bank.num_concepts; run_info["llm_dim"] = int(C_llm.shape[1])
print("Reproduction of released concept embeddings, cosine mean/min: plain", run_info["sfr_check_cos_plain_mean_min"],
      "| with instruction", run_info["sfr_check_cos_instruct_mean_min"])
subprocess.run(["rm", "-rf", "/tmp/hf/hub/models--Salesforce--SFR-Embedding-Mistral"])
gc.collect(); torch.cuda.empty_cache()
print("GPU memory in use after step 4 (GB):", [round(torch.cuda.memory_allocated(i)/1e9, 2) for i in range(torch.cuda.device_count())])
run_info["sec_prompt_embedding"] = round(time.time()-t)

In [ ]:
# 5. Load CLEAR and build the concept projection.
#    The authors' pipeline computes  normalise( (img @ norm(C_clip).T) @ C_llm ).
#    Because this is linear, it equals  normalise( img @ M ) with M = norm(C_clip).T @ C_llm,
#    which is computed once. Both forms are compared on real images in step 7.
from clear import load_pretrained, tokenize
dev = "cuda:0"
dev_cz = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
clear_model, clear_pre = load_pretrained(device=dev); clear_model.eval()
C_clip = torch.from_numpy(bank.ensure_clip_embeddings()).float()

# check that the released concept CLIP embeddings match CLEAR's text encoder (500 random concepts)
pick2 = np.random.default_rng(SEED+1).choice(bank.num_concepts, 500, replace=False)
with torch.no_grad():
    enc = F.normalize(clear_model.encode_text(tokenize([bank.concepts[i] for i in pick2]).to(dev)).float(), dim=-1).cpu()
cos_clip = (enc * F.normalize(C_clip[pick2], dim=-1)).sum(1)
run_info["concept_clip_check_cos_mean_min"] = [float(cos_clip.mean()), float(cos_clip.min())]
print("Released concept CLIP embeddings vs text encoder, cosine mean/min:", run_info["concept_clip_check_cos_mean_min"])
if cos_clip.min() < 0.999:
    print("Mismatch: recomputing all concept CLIP embeddings with the text encoder")
    chunks = []
    with torch.no_grad():
        for i in range(0, bank.num_concepts, 2048):
            chunks.append(clear_model.encode_text(tokenize(list(bank.concepts[i:i+2048])).to(dev)).float().cpu())
    C_clip = torch.cat(chunks); run_info["concept_clip_recomputed"] = True
else:
    run_info["concept_clip_recomputed"] = False

M = torch.zeros(C_clip.shape[1], C_llm.shape[1], dtype=torch.float64, device=dev)
C_llm_t = torch.from_numpy(C_llm)
for i in range(0, C_clip.shape[0], 8192):
    a = F.normalize(C_clip[i:i+8192].to(dev), dim=1).double()
    M += a.T @ C_llm_t[i:i+8192].to(dev).double()
M = M.float()
np.save(f"{OUT}/clear_concept_projection_M.npy", M.cpu().numpy())
pos_t = F.normalize(torch.from_numpy(pos_llm).to(dev), dim=-1); neg_t = F.normalize(torch.from_numpy(neg_llm).to(dev), dim=-1)
pos_ti = F.normalize(torch.from_numpy(pos_llm_i).to(dev), dim=-1); neg_ti = F.normalize(torch.from_numpy(neg_llm_i).to(dev), dim=-1)
with torch.no_grad():
    cpos = F.normalize(clear_model.encode_text(tokenize(POS).to(dev)).float(), dim=-1)
    cneg = F.normalize(clear_model.encode_text(tokenize(NEG).to(dev)).float(), dim=-1)
print("M:", tuple(M.shape))

In [ ]:
# 6. Load the 10 CheXzero checkpoints: from an attached Kaggle dataset if present, otherwise Google Drive
import gdown, clip
from torchvision.transforms import Compose, Normalize, Resize, InterpolationMode
def find_pts():
    hits = []
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if not d.startswith("images_")]
        hits += [os.path.join(root, f) for f in files if f.endswith(".pt")]
    return sorted(hits)
ckpts = find_pts()
if len(ckpts) < 10:
    os.makedirs("/kaggle/working/chexzero_weights", exist_ok=True)
    gdown.download_folder("https://drive.google.com/drive/folders/1makFLiEMbSleYltaRxw81aBhEDMpVwno",
                          output="/kaggle/working/chexzero_weights", quiet=True, remaining_ok=True)
    ckpts = sorted(glob.glob("/kaggle/working/chexzero_weights/**/*.pt", recursive=True))
assert len(ckpts) == 10, f"expected 10 CheXzero checkpoints, found {len(ckpts)}: {ckpts}"
print("CheXzero checkpoints from:", os.path.dirname(ckpts[0]))
run_info["chexzero_checkpoints"] = [[os.path.basename(c), os.path.getsize(c)] for c in ckpts]
cz_models, cz_text = [], []
for c in ckpts:
    m, _ = clip.load("ViT-B/32", device="cpu", jit=False)
    m.load_state_dict(torch.load(c, map_location="cpu")); m = m.to(dev_cz).eval()
    with torch.no_grad():
        tp = F.normalize(m.encode_text(clip.tokenize(POS, context_length=77).to(dev_cz)).float(), dim=-1)
        tn = F.normalize(m.encode_text(clip.tokenize(NEG, context_length=77).to(dev_cz)).float(), dim=-1)
    cz_models.append(m); cz_text.append((tp, tn))
print("CheXzero checkpoints loaded:", len(cz_models), "| GPU GB in use:", [round(torch.cuda.memory_allocated(i)/1e9, 2) for i in range(torch.cuda.device_count())])

In [ ]:
# 7. One pass over all Indiana images: CheXzero ensemble (CheXzero's own pad-to-square preprocessing),
#    CLEAR with padded images (primary) and with the package's squash resize (secondary)
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import ToTensor
def pad_square(img, size):                       # CheXzero data_process.preprocess
    w, h = img.size; r = float(size) / max(w, h)
    img = img.resize((int(w * r), int(h * r)), Image.LANCZOS)
    out = Image.new("L", (size, size)); out.paste(img, ((size - img.size[0]) // 2, (size - img.size[1]) // 2))
    return out
cz_tf = Compose([Normalize((101.48761,)*3, (83.43944,)*3), Resize(224, interpolation=InterpolationMode.BICUBIC)])
cl_norm = Normalize((101.48761/255,)*3, (83.43944/255,)*3)
class DS(Dataset):
    def __init__(self, paths): self.p = list(paths)
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        im = Image.open(self.p[i]).convert("L")
        g = np.asarray(pad_square(im, 320), dtype=np.float32)
        x_cz = cz_tf(torch.from_numpy(np.repeat(g[None], 3, axis=0)))
        x_pad = cl_norm(ToTensor()(pad_square(im, 448).convert("RGB")))
        x_sq = clear_pre(im.convert("RGB"))
        return x_cz, x_pad, x_sq
BS = 32
dl = DataLoader(DS(cohort.path), batch_size=BS, num_workers=4, pin_memory=True)
def pair_prob(f, tp, tn):
    p, n = f @ tp.T, f @ tn.T
    return torch.exp(p) / (torch.exp(p) + torch.exp(n))
R = {k: [] for k in ["chexzero", "clear_concept", "clear_concept_instr", "clear_direct", "clear_concept_squash"]}
t = time.time()
with torch.no_grad():
    for b, (x_cz, x_pad, x_sq) in enumerate(dl):
        x_cz = x_cz.to(dev_cz, non_blocking=True)
        per = [pair_prob(F.normalize(m.encode_image(x_cz).float(), dim=-1), tp, tn) for m, (tp, tn) in zip(cz_models, cz_text)]
        R["chexzero"].append(torch.stack(per, 0).mean(0).cpu().numpy())
        fi = F.normalize(clear_model.encode_image(x_pad.to(dev)).float(), dim=-1)
        rep_ = F.normalize(fi @ M, dim=-1)
        R["clear_concept"].append(pair_prob(rep_, pos_t, neg_t).cpu().numpy())
        R["clear_concept_instr"].append(pair_prob(rep_, pos_ti, neg_ti).cpu().numpy())
        R["clear_direct"].append(pair_prob(fi, cpos, cneg).cpu().numpy())
        fs = F.normalize(clear_model.encode_image(x_sq.to(dev)).float(), dim=-1)
        R["clear_concept_squash"].append(pair_prob(F.normalize(fs @ M, dim=-1), pos_t, neg_t).cpu().numpy())
        if b % 50 == 0: print(f"{min((b+1)*BS, len(cohort))}/{len(cohort)} images, {round((time.time()-t)/60,1)} min")
S = {k: np.concatenate(v).astype(np.float64) for k, v in R.items()}
for k, v in S.items():
    np.save(f"{OUT}/scores_{k}.npy", v.astype(np.float32))      # full precision, no rounding ties
run_info["min_scoring"] = round((time.time()-t)/60, 1)

In [ ]:
# 8. Analysis and verdict (decision rule in the first cell)
Y = cohort[FINDINGS].values.astype(np.float64)
uid_codes, uid_idx = np.unique(cohort.uid.values, return_inverse=True)
NU = len(uid_codes)
def wauc(s, y, w):
    o = np.argsort(s, kind="mergesort"); s, y, w = s[o], y[o], w[o]
    starts = np.flatnonzero(np.r_[True, s[1:] != s[:-1]])
    bp, bn = np.add.reduceat(w * y, starts), np.add.reduceat(w * (1 - y), starts)
    P, N = bp.sum(), bn.sum()
    if P == 0 or N == 0: return np.nan
    return float((bp * (np.cumsum(bn) - bn + 0.5 * bn)).sum() / (P * N))
rng = np.random.default_rng(SEED)
W = [np.bincount(rng.integers(0, NU, NU), minlength=NU).astype(np.float64) for _ in range(1000)]
def evaluate(mask):
    out = {}
    for k, s in S.items():
        pt = np.array([wauc(s[mask, j], Y[mask, j], np.ones(mask.sum())) for j in range(len(FINDINGS))])
        bt = np.array([[wauc(s[mask, j], Y[mask, j], w[uid_idx[mask]]) for j in range(len(FINDINGS))] for w in W])
        out[k] = (pt, bt)
    return out
def q(a): a = a[~np.isnan(a)]; return float(np.quantile(a, .025)), float(np.quantile(a, .975))
summary = {}
for name, mask in [("all_images", np.ones(len(cohort), bool)), ("frontal_only", (cohort.projection == "Frontal").values)]:
    ev = evaluate(mask)
    res = {k: {"macro": float(np.nanmean(pt)), "macro_ci95": q(np.nanmean(bt, axis=1))} for k, (pt, bt) in ev.items()}
    pc, bc = ev["chexzero"]
    for k in ["clear_concept", "clear_concept_instr", "clear_direct", "clear_concept_squash"]:
        pk, bk = ev[k]
        d, bd = float(np.nanmean(pk) - np.nanmean(pc)), np.nanmean(bk, axis=1) - np.nanmean(bc, axis=1)
        res[k]["delta_vs_chexzero"] = d; res[k]["delta_ci95"] = q(bd)
        res[k]["wins_losses_vs_chexzero"] = [int((pk > pc).sum()), int((pk < pc).sum())]
    res["per_finding"] = {f: {k: round(float(ev[k][0][j]), 3) for k in S} for j, f in enumerate(FINDINGS)}
    summary[name] = res
p = summary["all_images"]["clear_concept"]
lo, hi = p["delta_ci95"]; wins, losses = p["wins_losses_vs_chexzero"]
if p["delta_vs_chexzero"] > 0 and wins > losses: verdict = "CONSISTENT with the CLEAR paper (CLEAR ahead of CheXzero)"
elif p["delta_vs_chexzero"] < 0 and hi < 0: verdict = "INCONSISTENT with the CLEAR paper (CLEAR behind CheXzero, interval excludes 0)"
else: verdict = "INCONCLUSIVE"
summary["verdict"] = verdict
json.dump({"run_info": run_info, "summary": summary}, open(f"{OUT}/indiana_results.json", "w"), indent=1)
pd.DataFrame(summary["all_images"]["per_finding"]).T.to_csv(f"{OUT}/per_finding_auroc.csv")
!cd /kaggle/working && rm -f indiana_outputs.zip && zip -qr indiana_outputs.zip indiana_out
print("===== INDIANA SUMMARY =====")
print("images", run_info["n_images"], "| reports", run_info["n_reports"], "| frontal", run_info["n_frontal"], "| findings", len(FINDINGS))
print("SFR check plain/instr:", run_info.get("sfr_check_cos_plain_mean_min"), run_info.get("sfr_check_cos_instruct_mean_min"))
for name in ["all_images", "frontal_only"]:
    print(f"-- {name}")
    for k in S:
        r = summary[name][k]; line = f"  {k:<22} macro {r['macro']:.3f} ({r['macro_ci95'][0]:.3f}-{r['macro_ci95'][1]:.3f})"
        if "delta_vs_chexzero" in r:
            line += f" | minus CheXzero {r['delta_vs_chexzero']:+.3f} ({r['delta_ci95'][0]:+.3f} to {r['delta_ci95'][1]:+.3f}) | wins/losses {r['wins_losses_vs_chexzero']}"
        print(line)
print("-- per finding (all images): chexzero / clear_concept / clear_direct")
for f, v in summary["all_images"]["per_finding"].items():
    print(f"  {f:<40} {v['chexzero']:.3f} / {v['clear_concept']:.3f} / {v['clear_direct']:.3f}")
print("VERDICT:", verdict)
print("===== END =====")